# FoX restricted experiment: learned versus frozen retrieval forgetting

Run **Runtime → Run all** to train SGD and Adam on the paper's exact finite pair population,
then plot the model's **correct-answer probability at different test lags**.
The first settings cell controls vocabulary size **N**, training recall lag **R**, seeds and compute.
All source and checks are included below: no repository, data upload, API key or prior result is needed.

The default is a modest experiment: N=8, R=2, width 2,048, three paired seeds,
12,000 total updates (including acquisition), and four arms. A GPU runtime is useful; CPU also works.
Set `SMOKE=True` for a short software check, whose results are not evidence for asymptotic theory.
Outputs, configurations, source and plots are saved and offered as one downloadable ZIP.

This is the paper's **restricted two-layer answer-supervised architecture** with its parser masks,
local binder, trainable raw query/key tables and scalar decoder. It is an empirical sanity check
of those dynamics, rather than a language-model benchmark. A finite run can support or challenge
the predicted trend; it cannot certify asymptotic or uniform generalization.


## The finite dataset and the meaning of lag R

Let K=N(N−1). We use **every ordered pair of distinct keys** a≠b. For each pair,

\[
O^{a,b}=(a,-1),(a,-1),(b,-1),(a,+1),?a,
\]
\[
C_R^{a,b}=(a,+1),\underbrace{(b,-1),\ldots,(b,-1)}_{R-1},?a.
\]

Both templates include their global value complements. Every key additionally has the two
one-record calibration examples. Therefore K is the number of **pair identities**;
the full supervised population contains **4K+2N serialized sequences**, not K sequences total.
The objective weights the overwrite, recall and calibration families by 0.2, 0.2 and 0.6.
SGD samples pair identities with replacement and evaluates both weighted families; calibration
is evaluated exactly. Adam uses the full pair population.

**Lags count records, with the newest record at lag 1.** Only recall training changes when R
changes; the four-record overwrite template stays fixed. Training uses one recall range R,
not a mixture of every shorter range. All losses supervise the final binary answer only.

Evaluation keeps the same vocabulary and tests a target at each requested lag with newer,
opposite-valued distractors. Additional old, opposite-valued same-key prefixes challenge overwrite
robustness. The plots report the probability assigned by the scalar decoder to the correct answer,
not merely target attention or thresholded classification accuracy.


## The four arms and predictions to examine

| Arm | Local binding penalty g | Retrieval penalty h | Optimizer |
|---|---|---|---|
| Learned / SGD | learned | learned | SGD |
| Learned / Adam | learned | learned | full-batch annealed-epsilon Adam |
| Frozen retrieval / SGD | learned | fixed h₀ | SGD |
| Frozen retrieval / Adam | learned | fixed h₀ | full-batch annealed-epsilon Adam |

The frozen intervention fixes **only retrieval forgetting**. The binder must still learn.
An optional `both_frozen` control is a separate, obstructed case. The learned and frozen arms
start with identical h₀ and matched raw table draws for each seed.

At R=2, the learned-gate SGD theorem gives a worst-case cutoff at lag 4; learned-gate Adam
has an expanding range, linear in its cumulative scalar learning-rate clock under the theorem's
conditions. With frozen retrieval h₀>log 2, the answer-supervised extension predicts an expanding
range for both optimizers; its Adam range is quadratic in that clock. Frozen retrieval can
produce confident correct answers even though target attention retains an overwrite error floor.

For R>2, the learned-SGD extension predicts cutoff R+2 under its basin/preparation and
continuation conditions. This notebook does not implement the extension's extra finite
preparation construction. The frozen-retrieval convergence proof in the accompanying theory
uses R=2. Setting R=4 is a useful extension experiment, but must not be described as a direct
empirical verification of every proved hypothesis.

The theorem's very conservative SGD existence schedule is not a practical Colab schedule.
We use an explicit finite polynomial schedule with increasing pair batches and a positive,
summable table-rate tail. The plots retain diagnostics needed to assess this approximation.
A curve that has not reached its predicted regime is inconclusive, and a mean curve can conceal
a hard key pair. Inspect minimum probabilities, individual seeds, training loss and gap diagnostics.


In [ ]:
# USER SETTINGS — edit this cell, then run all.
from pathlib import Path
from datetime import datetime, timezone
import importlib.util
import os
import sys
import tempfile

SMOKE = False
N = 8
R = 2                         # Try R=4 separately; read the extension caveat above.
WIDTH = 2048
STEPS = 12000                 # Total updates, including three acquisition updates.
SEEDS = (0, 1, 2)              # Paired across both optimizers and gate modes.
H0 = 1.0                      # Penalty PER RECORD. Default exceeds log(2).
EVAL_LAGS = tuple(range(1, 33))
PREFIXES = (0, 16, 64)
INCLUDE_BOTH_FROZEN = False
LOG_EVERY = 1000
RUN_QA = True
RUN_TRAINING = True
DOWNLOAD_AT_END = True
USE_DRIVE = False
RUN_TAG = "fox_restricted_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

# Colab normally includes these numerical packages. Elsewhere install requirements.txt.
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "fox_restricted_mpl"))
import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import display, Image, FileLink, Markdown

def show_table(columns, rows):
    def clean(value):
        return str(value).replace("|", "&#124;").replace("\n", " ")
    lines = ["| " + " | ".join(map(clean, columns)) + " |",
             "| " + " | ".join("---" for _ in columns) + " |"]
    lines += ["| " + " | ".join(map(clean, row)) + " |" for row in rows]
    display(Markdown("\n".join(lines)))

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except (ModuleNotFoundError, ValueError):
    IN_COLAB = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(1)       # Tiny matrix operations are faster without thread-pool overhead.
if DEVICE == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
if SMOKE:
    N, WIDTH, STEPS, SEEDS = 4, 64, 20, (0,)
    EVAL_LAGS, PREFIXES, LOG_EVERY = tuple(range(1, 9)), (0, 4), 10

if USE_DRIVE:
    if not IN_COLAB:
        raise RuntimeError("USE_DRIVE applies to Colab; set OUTPUT_PARENT to a local path instead.")
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_PARENT = Path("/content/drive/MyDrive/fox_restricted_runs")
else:
    OUTPUT_PARENT = Path("/content/fox_restricted_runs") if IN_COLAB else Path.cwd() / "fox_restricted_runs"
OUT_DIR = OUTPUT_PARENT / (RUN_TAG + ("_smoke" if SMOKE else ""))
CODE_DIR = Path(tempfile.mkdtemp(prefix="fox_restricted_source_"))
SOURCES = {}
print("Device:", DEVICE, "| PyTorch:", torch.__version__, "| Outputs:", OUT_DIR)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name())
print(f"N={N}, R={R}; {N*(N-1)} pair identities; {4*N*(N-1)+2*N} serialized sequences")
print(f"{len(SEEDS) * (3 if INCLUDE_BOTH_FROZEN else 2) * 2} runs × {STEPS:,} total updates")


## Embedded implementation

These folded cells contain the complete Python modules. Their SHA-256 hashes are saved with the
outputs, and the tests run before training. The implementation uses float64 signed-log gradients
and signed-log Adam moments to retain small derivatives without squaring them to zero.
This is the ordinary answer objective and actual Adam update, with an independent native
float64 cross-check in the preflight suite. Signed-log arithmetic still has finite precision.


In [ ]:
SOURCES['core'] = '"""Restricted ordinary-answer FoX experiment, implemented in float64.\n\nThe analytic backend evaluates the exact finite data objective. Signed-log\ngradients and Adam buffers retain very small derivatives without substituting\npointer supervision, sign descent, gradient clipping, or an epsilon floor.\n``dense_stream`` independently materializes both attention heads for auditing.\n"""\nfrom dataclasses import asdict, dataclass\nimport json\nimport math\nfrom pathlib import Path\n\nimport torch\nfrom torch.nn import functional as F\n\n\n@dataclass\nclass Config:\n    n: int = 8\n    d: int = 2048\n    R: int = 2\n    steps: int = 12000\n    seed: int = 0\n    device: str = "cpu"\n    sigma: float = 1e-6\n    gamma: float = 1 / 16\n    c0: float = 1.\n    D: float = 0.\n    kcal: float = .6\n    wo: float = .2\n    wc: float = .2\n    q0: float = .5\n    u0: float = 1.7\n    h0: float = 1.\n    x0: float | None = None\n    gate_mode: str = "learned"\n    am: float = 1.\n    ag: float = 1.5\n    ax: float = .05\n    aw: float = .1\n    beta1: float = .9\n    beta2: float = .999\n    eps_decay: float = .1\n    lr_adam: float = .015\n    lr_sgd: float = 5.\n    offset: float = 1000.\n    power: float = .75\n    table_lr: float = 1e-9\n    table_power: float = 2.\n    pair_batch: int = 4096\n    batch_growth: float = .5\n    log_points: int = 80\n    eval_max_lag: int = 128\n    max_seconds: float = 3600.\n\n    def __post_init__(self):\n        for name, minimum in (("n", 2), ("d", 1), ("R", 2), ("steps", 0),\n                              ("pair_batch", 1), ("log_points", 2), ("eval_max_lag", 1)):\n            value = getattr(self, name)\n            if isinstance(value, bool) or not isinstance(value, int) or value < minimum:\n                raise ValueError(f"{name} must be an integer >= {minimum}")\n        if self.gate_mode not in ("learned", "retrieval_frozen", "both_frozen"):\n            raise ValueError("gate_mode must be learned, retrieval_frozen, or both_frozen")\n        positive = ("sigma", "gamma", "c0", "q0", "u0", "h0", "kcal", "wo", "wc",\n                    "am", "ag", "ax", "aw", "lr_adam", "lr_sgd", "offset", "table_lr")\n        if any(not math.isfinite(getattr(self, name)) or getattr(self, name) <= 0 for name in positive):\n            raise ValueError("Initialization scales, mixture weights, and learning rates must be positive and finite")\n        if not self.kcal > .5 or not math.isclose(self.kcal + self.wo + self.wc, 1., abs_tol=1e-12):\n            raise ValueError("Require kcal > 1/2 and kcal + wo + wc = 1")\n        if not 0 <= self.beta1 < 1 or not self.beta1 ** 2 < self.beta2 < 1:\n            raise ValueError("Adam requires 0 <= beta1 < 1 and beta1**2 < beta2 < 1")\n        if not 2 / 3 < self.power <= 1 or not self.table_power > 1:\n            raise ValueError("Use scalar power in (2/3, 1] and summable table power > 1")\n        if self.eps_decay < 0 or self.batch_growth < 0:\n            raise ValueError("epsilon decay and batch growth must be nonnegative")\n        if self.x0 is None:\n            self.x0 = self.h0 + math.log(-math.expm1(-self.h0))\n        if not math.isfinite(self.x0) or not math.isfinite(self.D):\n            raise ValueError("x0 and D must be finite")\n        effective_h = max(self.x0, 0.) + math.log1p(math.exp(-abs(self.x0)))\n        if not math.isclose(effective_h, self.h0, rel_tol=1e-12, abs_tol=1e-12):\n            raise ValueError("x0 must equal inverse_softplus(h0); use replace(cfg, h0=new_h, x0=None) to change the slope")\n\n\ndef frozen_scalar_indices(config):\n    return {"learned": (), "retrieval_frozen": (4,), "both_frozen": (2, 3, 4)}[config.gate_mode]\n\n\ndef active_parameter_count(config):\n    return 2 * config.n * config.d + 6 - len(frozen_scalar_indices(config))\n\n\ndef slog(x):\n    return torch.sign(x), torch.log(torch.abs(x))\n\n\ndef sadd(s1, l1, s2, l2):\n    """Signed log addition; exact cancellation has sign zero and log -inf."""\n    same = s1 == s2\n    hi = torch.maximum(l1, l2)\n    diff = torch.abs(l1 - l2)\n    difference = hi + torch.log(-torch.expm1(-diff))\n    logabs = torch.where(same, torch.logaddexp(l1, l2), difference)\n    sign = torch.where(same, s1, torch.where(l1 > l2, s1, s2))\n    zero = ((l1 == l2) & ~same) | (torch.isneginf(l1) & torch.isneginf(l2))\n    return torch.where(zero, 0., sign), torch.where(zero, -torch.inf, logabs)\n\n\ndef ssum(signs, logs, dim=None):\n    if dim is None:\n        signs, logs, dim = signs.reshape(-1), logs.reshape(-1), 0\n    pos = torch.logsumexp(torch.where(signs > 0, logs, -torch.inf), dim)\n    neg = torch.logsumexp(torch.where(signs < 0, logs, -torch.inf), dim)\n    return sadd(torch.ones_like(pos), pos, -torch.ones_like(neg), neg)\n\n\ndef logsoftplus(x):\n    # For x < -30, the relative difference from exp(x) is < 5e-14.\n    return torch.where(x < -30, x, torch.log(F.softplus(x)))\n\n\ndef quantities_from_theta(config, theta, detach_frozen=False):\n    q, p, u, v, x, w = theta.unbind()\n    if detach_frozen:\n        if config.gate_mode == "both_frozen":\n            u, v = u.detach(), v.detach()\n        if config.gate_mode != "learned":\n            x = x.detach()\n    m, g, h = config.c0 * q * p, F.softplus(u * v), F.softplus(x)\n    lrho = F.logsigmoid(config.D - 2 * g)\n    rho = torch.exp(lrho)\n    return q, p, u, v, x, w, m, g, h, rho, lrho\n\n\ndef recall_terms(z, m, h, rho, lag):\n    """One term per wrong record, in target-relative log-weight units."""\n    if lag < 2:\n        raise ValueError("Recall training lag must be >= 2")\n    first = -z * m * (1 - rho) + h\n    if lag == 2:\n        return first.unsqueeze(-1)\n    offsets = torch.arange(2, lag, dtype=z.dtype, device=z.device)\n    pure = (-z * m).unsqueeze(-1) + offsets * h\n    return torch.cat((first.unsqueeze(-1), pure), dim=-1)\n\n\ndef _pair_log_mass(z, counts, validate=True):\n    if counts is None:\n        return torch.full_like(z, -math.log(z.numel()))\n    counts = torch.as_tensor(counts, device=z.device, dtype=z.dtype)\n    if validate:\n        if counts.shape != z.shape or bool((counts < 0).any()) or not bool(counts.sum() > 0):\n            raise ValueError("counts must be a nonnegative vector with positive total, one entry per ordered pair")\n        if not bool(torch.isfinite(counts).all()):\n            raise ValueError("counts must be finite")\n    return torch.log(counts) - torch.log(counts.sum())\n\n\nclass Model:\n    def __init__(self, cfg):\n        self.cfg = cfg\n        generator = torch.Generator(device=cfg.device).manual_seed(cfg.seed)\n        self.Q = torch.randn(cfg.n, cfg.d, generator=generator, device=cfg.device, dtype=torch.float64) * cfg.sigma\n        self.K = torch.randn(cfg.n, cfg.d, generator=generator, device=cfg.device, dtype=torch.float64) * cfg.sigma\n        self.theta = torch.tensor([cfg.q0, cfg.q0, cfg.u0, cfg.u0, cfg.x0, 0.],\n                                  device=cfg.device, dtype=torch.float64)\n        self.mask = ~torch.eye(cfg.n, device=cfg.device, dtype=torch.bool)\n        self.aa, self.bb = torch.where(self.mask)\n        self.frozen_scalar_indices = frozen_scalar_indices(cfg)\n\n    def params(self):\n        return [self.Q, self.K, self.theta]\n\n    def gaps(self):\n        scores = self.Q @ self.K.T\n        return scores.diag()[:, None] - scores\n\n    def quantities(self):\n        return quantities_from_theta(self.cfg, self.theta)\n\n    @property\n    def active_parameter_count(self):\n        return active_parameter_count(self.cfg)\n\n    def rows(self, z):\n        q, p, u, v, x, w, m, g, h, rho, lrho = self.quantities()\n        terms = torch.stack((z * m * rho - 2 * h, z * m * rho - 3 * h,\n                             -z * m * (1 - 2 * rho) - h), dim=-1)\n        lo = torch.logsumexp(terms, dim=-1)\n        lc = torch.logsumexp(recall_terms(z, m, h, rho, self.cfg.R), dim=-1)\n        return terms, lo, lc, -torch.tanh(lo / 2), -torch.tanh(lc / 2)\n\n    @torch.no_grad()\n    def gradients(self, counts=None, validate_counts=True):\n        """Exact ordinary-loss derivatives in sign/log-absolute representation.\n\n        Pair counts give a stratified estimator: calibration is exact and both\n        overwrite and recall losses are evaluated for every sampled pair.\n        ``validate_counts=False`` is only for internally generated multinomial\n        counts. It removes host synchronizations from the trusted GPU path.\n        """\n        c = self.cfg\n        q, p, u, v, x, w, m, g, h, rho, lrho = self.quantities()\n        z = self.gaps()[self.aa, self.bb]\n        terms, lo, lc, so, sc = self.rows(z)\n        rterms = recall_terms(z, m, h, rho, c.R)\n        lp = _pair_log_mass(z, counts, validate=validate_counts)\n        ltw = torch.log(2 * torch.abs(w))\n        common_o = lp + math.log(c.wo) + ltw + F.logsigmoid(-w * so) - 2 * F.softplus(lo)\n        common_c = lp + math.log(c.wc) + ltw + F.logsigmoid(-w * sc) - 2 * F.softplus(lc)\n        la, lb, le = (common_o[:, None] + terms).unbind(-1)\n        recall_pressure = common_c[:, None] + rterms\n        lf = recall_pressure[:, 0]\n        # P is the derivative with respect to M=z*m, divided by sign(w).\n        logs_p = torch.cat((torch.stack((la + lrho, lb + lrho,\n                                        le + torch.log(torch.abs(1 - 2 * rho)),\n                                        lf + torch.log1p(-rho)), dim=-1),\n                            recall_pressure[:, 1:]), dim=-1)\n        signs_p = -torch.ones_like(logs_p)\n        signs_p[:, :2] = 1.\n        signs_p[:, 2] = -torch.sign(1 - 2 * rho)\n        ps, pl = ssum(signs_p, logs_p, dim=-1)\n        ps = ps * torch.sign(w)\n        dzs, dzl = ps * torch.sign(m), pl + torch.log(torch.abs(m))\n        shift = torch.max(dzl)\n        shift = torch.where(torch.isfinite(shift), shift, torch.zeros_like(shift))\n        weights = torch.zeros(c.n, c.n, device=z.device, dtype=z.dtype)\n        weights[self.aa, self.bb] = dzs * torch.exp(dzl - shift)\n        score_grad = -weights\n        score_grad.diagonal().add_(weights.sum(dim=1))\n        sq, lq = slog(score_grad @ self.K)\n        sk, lk = slog(score_grad.T @ self.Q)\n        sm, lm = ssum(ps * torch.sign(z), pl + torch.log(torch.abs(z)))\n        rec_offsets = torch.arange(1, c.R, dtype=z.dtype, device=z.device)\n        logs_h = torch.cat((torch.stack((la + math.log(2), lb + math.log(3), le), dim=-1),\n                            recall_pressure + rec_offsets.log()), dim=-1)\n        signs_h = torch.ones_like(logs_h)\n        signs_h[:, :3] = -1.\n        sh, lh = ssum(signs_h * torch.sign(w), logs_h)\n        # Only the first recall distractor\'s binder leaks the queried key.\n        binder_pressure = torch.logsumexp(torch.stack((la, lb, le + math.log(2), lf), dim=-1), dim=-1)\n        sg, lg = ssum(-torch.sign(z * m * w), binder_pressure + torch.log(torch.abs(z * m))\n                     + math.log(2) + lrho + torch.log1p(-rho))\n        sw, lw = ssum(torch.cat((-torch.ones(1, device=z.device, dtype=z.dtype), -torch.sign(so), -torch.sign(sc))),\n                       torch.cat(((math.log(c.kcal) + F.logsigmoid(-w)).reshape(1),\n                                  lp + math.log(c.wo) + torch.log(torch.abs(so)) + F.logsigmoid(-w * so),\n                                  lp + math.log(c.wc) + torch.log(torch.abs(sc)) + F.logsigmoid(-w * sc))))\n        st = torch.stack((sm * torch.sign(p), sm * torch.sign(q), sg * torch.sign(v),\n                          sg * torch.sign(u), sh, sw))\n        lt = torch.stack((lm + torch.log(torch.abs(c.c0 * p)), lm + torch.log(torch.abs(c.c0 * q)),\n                          lg + torch.log(torch.abs(v)) + F.logsigmoid(u * v),\n                          lg + torch.log(torch.abs(u)) + F.logsigmoid(u * v),\n                          lh + F.logsigmoid(x), lw))\n        if self.frozen_scalar_indices:\n            indices = list(self.frozen_scalar_indices)\n            st[indices], lt[indices] = 0., -torch.inf\n        logloss = torch.logsumexp(torch.cat(((math.log(c.kcal) + logsoftplus(-w)).reshape(1),\n                                             lp + math.log(c.wo) + logsoftplus(-w * so),\n                                             lp + math.log(c.wc) + logsoftplus(-w * sc))), dim=0)\n        finite = torch.isfinite(dzl)\n        high = torch.where(finite, dzl, -torch.inf).max()\n        low = torch.where(finite, dzl, torch.inf).min()\n        span = torch.where(finite.any(), high - low, torch.zeros_like(w))\n        return [(sq, lq + shift), (sk, lk + shift), (st, lt)], {"logloss": logloss, "table_log_span": span}\n\n\ndef native_loss(model, counts=None, params=None):\n    """Conventional PyTorch autograd loss, used as an independent derivative check.\n\n    This path may underflow at extreme checkpoints; it never substitutes a\n    log-gradient result when native arithmetic loses a derivative.\n    """\n    Q, K, theta = model.params() if params is None else params\n    c = model.cfg\n    q, p, u, v, x, w, m, g, h, rho, lrho = quantities_from_theta(c, theta, detach_frozen=True)\n    scores = Q @ K.T\n    z = (scores.diag()[:, None] - scores)[model.mask]\n    log_o = torch.logsumexp(torch.stack((z * m * rho - 2 * h, z * m * rho - 3 * h,\n                                       -z * m * (1 - 2 * rho) - h), dim=-1), dim=-1)\n    log_c = torch.logsumexp(recall_terms(z, m, h, rho, c.R), dim=-1)\n    loss = c.wo * F.softplus(w * torch.tanh(log_o / 2)) + c.wc * F.softplus(w * torch.tanh(log_c / 2))\n    return c.kcal * F.softplus(-w) + (loss * _pair_log_mass(z, counts).exp()).sum()\n\n\ndef native_gradients(model, counts=None):\n    with torch.enable_grad():\n        params = [p.detach().clone().requires_grad_(True) for p in model.params()]\n        loss = native_loss(model, counts, params)\n        gradients = [p.detach() for p in torch.autograd.grad(loss, params)]\n    return gradients, {"loss": loss.detach(), "logloss": loss.detach().log(),\n                       "native_loss_underflow": bool(loss.detach() == 0)}\n\n\ndef dense_stream(params, config, keys, values, query, label=1.):\n    """Literal two-head forward pass on serialized key/value record inputs.\n\n    Returns ordinary answer loss, record attention, and correct-answer logit.\n    The binder scores raw-token offsets 1 and 3. Retrieval uses slope h/2 per\n    raw token, equivalent to h per record; the query is never a candidate.\n    """\n    Q, K, theta = params\n    q, p, u, v, x, w = theta.unbind()\n    if config.gate_mode == "both_frozen":\n        u, v = u.detach(), v.detach()\n    if config.gate_mode != "learned":\n        x = x.detach()\n    g, h = F.softplus(u * v), F.softplus(x)\n    keys = torch.as_tensor(keys, device=Q.device, dtype=torch.long)\n    values = torch.as_tensor(values, device=Q.device, dtype=Q.dtype)\n    if keys.ndim != 1 or values.shape != keys.shape or keys.numel() < 1:\n        raise ValueError("A stream needs equally sized nonempty key/value vectors")\n    if bool(((keys < 0) | (keys >= config.n)).any()) or not 0 <= int(query) < config.n:\n        raise ValueError("Key IDs must be in the configured vocabulary")\n    if not bool(((values == -1) | (values == 1)).all()) or float(label) not in (-1., 1.):\n        raise ValueError("Values and answer label must be -1 or +1")\n    vectors = K[keys]\n    binding_weights = torch.stack((-g, config.D - 3 * g)).softmax(dim=0)\n    bound = torch.cat((vectors[:1], binding_weights[0] * vectors[1:] + binding_weights[1] * vectors[:-1]), dim=0)\n    raw_distances = 2 * torch.arange(keys.numel(), 0, -1, device=Q.device, dtype=Q.dtype)\n    scores = config.c0 * q * p * (bound @ Q[int(query)]) - (h / 2) * raw_distances\n    attention = scores.softmax(dim=0)\n    correct_logit = label * w * torch.sum(attention * values)\n    return F.softplus(-correct_logit), attention, correct_logit\n\n\ndef dense_objective(model, counts=None, params=None):\n    """Slow literal-stream objective, including both label complements."""\n    params = model.params() if params is None else params\n    c = model.cfg\n    rows = []\n    for a in range(c.n):\n        for b in range(c.n):\n            if a == b:\n                continue\n            o = sum(dense_stream(params, c, [a, a, b, a], [-y, -y, -y, y], a, y)[0] for y in (-1, 1)) / 2\n            rec = sum(dense_stream(params, c, [a] + [b] * (c.R - 1), [y] + [-y] * (c.R - 1), a, y)[0] for y in (-1, 1)) / 2\n            rows.append(c.wo * o + c.wc * rec)\n    rows = torch.stack(rows)\n    calibration = torch.stack([dense_stream(params, c, [a], [y], a, y)[0]\n                               for a in range(c.n) for y in (-1, 1)]).mean()\n    return c.kcal * calibration + (rows * _pair_log_mass(rows, counts).exp()).sum()\n\n\nclass LogAdam:\n    """Bias-corrected Adam with signed-log first and log second moments.\n\n    All buffers start at zero. Moment histories persist across acquisition and\n    continuation. Only storage arithmetic differs from conventional Adam.\n    """\n    def __init__(self, params, b1, b2):\n        if not 0 <= b1 < 1 or not b1 ** 2 < b2 < 1:\n            raise ValueError("Require 0 <= beta1 < 1 and beta1**2 < beta2 < 1")\n        self.b1, self.b2, self.t = b1, b2, 0\n        self.ms = [torch.zeros_like(p) for p in params]\n        self.ml = [torch.full_like(p, -torch.inf) for p in params]\n        self.vl = [torch.full_like(p, -torch.inf) for p in params]\n\n    @torch.no_grad()\n    def step(self, params, grads, rates, logeps):\n        self.t += 1\n        correction_m = math.log1p(-self.b1 ** self.t)\n        correction_v = math.log1p(-self.b2 ** self.t)\n        directions = []\n        for i, (parameter, (sign, logabs), rate) in enumerate(zip(params, grads, rates)):\n            self.ms[i], self.ml[i] = sadd(self.ms[i], self.ml[i] + (math.log(self.b1) if self.b1 else -math.inf),\n                                          sign, logabs + math.log1p(-self.b1))\n            self.vl[i] = torch.logaddexp(self.vl[i] + math.log(self.b2), 2 * logabs + math.log1p(-self.b2))\n            denominator = torch.logaddexp(.5 * (self.vl[i] - correction_v), torch.full_like(parameter, logeps))\n            direction = torch.where(self.ms[i] == 0, 0., self.ms[i] * torch.exp(self.ml[i] - correction_m - denominator))\n            parameter.add_(-rate * direction)\n            directions.append(direction)\n        return directions\n\n\n@torch.no_grad()\ndef gradient_step(model, grads, rates):\n    for parameter, (sign, logabs), rate in zip(model.params(), grads, rates):\n        parameter.add_(-rate * sign * torch.exp(logabs))\n\n\ndef _mask_rates(config, scalar_rates):\n    scalar_rates[list(frozen_scalar_indices(config))] = 0.\n    return scalar_rates\n\n\ndef rates_at(c, kind, t):\n    if kind not in ("sgd", "adam", "adam_annealed", "adam_fixed"):\n        raise ValueError("kind must be sgd, adam, adam_annealed, or adam_fixed")\n    base = (c.lr_adam if kind.startswith("adam") else c.lr_sgd) * (1 + t / c.offset) ** (-c.power)\n    table = c.table_lr * (1 + t / c.offset) ** (-c.table_power)\n    multipliers = _mask_rates(c, torch.tensor([c.am, c.am, c.ag, c.ag, c.ax, c.aw], device=c.device, dtype=torch.float64))\n    return base, [table, table, base * multipliers]\n\n\ndef acquisition_rates(c, kind):\n    """Three ordinary-loss updates with deterministic rates fixed before draws.\n\n    SGD rates use the exact zero-gap objective at this R and initialization.\n    These practical finite constants do not instantiate the proof\'s unspecified\n    sufficiently-small bounds or the extra R>2 basin preparation protocol.\n    """\n    if kind.startswith("adam"):\n        theta = _mask_rates(c, torch.tensor([1e-5] * 5 + [.005], device=c.device, dtype=torch.float64))\n        return [[c.gamma / math.sqrt(c.d)] * 2 + [theta.clone()] for _ in range(3)]\n    if kind != "sgd":\n        raise ValueError("Acquisition supports sgd or adam")\n    probe = Model(c)\n    probe.Q.zero_()\n    probe.K.zero_()\n    sw, lw = probe.gradients()[0][-1]\n    fw = float(sw[-1] * lw[-1].exp())\n    if not math.isfinite(fw) or fw >= 0:\n        raise ValueError(f"Initial decoder derivative must be negative; got {fw}")\n    probe.theta[-1] = .2\n    with torch.enable_grad():\n        z = torch.zeros((), device=c.device, dtype=torch.float64, requires_grad=True)\n        _, _, _, so, sc = probe.rows(z)\n        phi = c.wo * F.softplus(-.2 * so) + c.wc * F.softplus(-.2 * sc)\n        dstar = -float(torch.autograd.grad(phi, z)[0])\n    if not math.isfinite(dstar) or dstar <= 0:\n        raise ValueError(f"Acquisition requires a positive zero-gap matching signal; dstar={dstar}")\n    astar = dstar / (c.n - 1)\n    theta = _mask_rates(c, torch.tensor([1e-7] * 5 + [1e-4], device=c.device, dtype=torch.float64))\n    first = theta.clone()\n    first[-1] = .2 / (-fw)\n    return [[1e-7, 1e-7, first],\n            [c.gamma / (astar * c.sigma * math.sqrt(c.d))] * 2 + [theta.clone()],\n            [1 / astar] * 2 + [theta.clone()]]\n\n\ndef _geom_log(count, h):\n    if count == 0:\n        return torch.full_like(h, -torch.inf)\n    if float(h) == 0:\n        return torch.full_like(h, math.log(count))\n    return torch.log(-torch.expm1(-count * h)) - torch.log(-torch.expm1(-h))\n\n\ndef boundary_log_odds(gaps, m, h, rho, lag, prefix_count):\n    """Exact witness odds: stale a-prefix, correct a, then lag-1 wrong b\'s.\n\n    ``prefix_count=math.inf`` gives the infinite-prefix limit analytically.\n    """\n    if not isinstance(lag, int) or lag < 1 or prefix_count < 0:\n        raise ValueError("lag must be a positive integer and prefix_count nonnegative")\n    result = (-h + _geom_log(prefix_count, h)).expand_as(gaps)\n    if lag >= 2:\n        result = torch.logaddexp(result, -m * (1 - rho) * gaps + h)\n    if lag >= 3:\n        result = torch.logaddexp(result, -m * gaps + (lag - 1) * h + _geom_log(lag - 2, h))\n    return result\n\n\ndef _sequence(keys, values, query, answer, family, pair_id=None):\n    tokens = []\n    for key, value in zip(keys, values):\n        tokens.extend((f"key_{key}", f"value_{value:+d}"))\n    tokens.extend(("?", f"key_{query}"))\n    matching = [i for i, key in enumerate(keys) if key == query]\n    target = matching[-1]\n    return {"family": family, "pair_id": pair_id, "keys": keys, "values": values,\n            "query": query, "answer": answer, "target_lag": len(keys) - target,\n            "tokens": tokens}\n\n\ndef pair_cases(config):\n    """Exactly N(N-1) pair cases; each contains four supervised sequences."""\n    result = []\n    for a in range(config.n):\n        for b in range(config.n):\n            if a == b:\n                continue\n            pair_id = len(result)\n            overwrite = [_sequence([a, a, b, a], [-y, -y, -y, y], a, y, "overwrite", pair_id) for y in (-1, 1)]\n            recall = [_sequence([a] + [b] * (config.R - 1), [y] + [-y] * (config.R - 1),\n                                a, y, "recall", pair_id) for y in (-1, 1)]\n            result.append({"pair_id": pair_id, "query_key": a, "distractor_key": b,\n                           "overwrite": overwrite, "recall": recall})\n    return result\n\n\ndef export_dataset(config, directory):\n    """Export the pair-indexed data and its fully expanded weighted sequences."""\n    directory = Path(directory)\n    directory.mkdir(parents=True, exist_ok=True)\n    cases = pair_cases(config)\n    calibration = [_sequence([a], [y], a, y, "calibration") for a in range(config.n) for y in (-1, 1)]\n    sequences = []\n    for case in cases:\n        for family, weight in (("overwrite", config.wo), ("recall", config.wc)):\n            for row in case[family]:\n                sequences.append({**row, "population_weight": weight / (2 * len(cases))})\n    sequences.extend({**row, "population_weight": config.kcal / len(calibration)} for row in calibration)\n    manifest = {"key_vocabulary_size": config.n, "pair_cases": len(cases), "recall_lag": config.R,\n                "base_recall_sequences": len(cases), "base_overwrite_sequences": len(cases),\n                "paired_sequences_with_complements": 4 * len(cases),\n                "calibration_sequences": len(calibration), "total_expanded_sequences": len(sequences),\n                "mixture_weights": {"calibration": config.kcal, "overwrite": config.wo, "recall": config.wc},\n                "lag_units": "records, newest record at lag 1", "query_is_attention_candidate": False,\n                "dataset_items": "N(N-1) pair cases; each evaluates both families and complements",\n                "files": {"pair_cases": "pairs.jsonl", "expanded_sequences": "dataset.jsonl",\n                          "base_recall_at_R": "recall_at_R.jsonl", "pair_index": "pair_index.csv"}}\n    (directory / "pairs.jsonl").write_text("".join(json.dumps(case) + "\\n" for case in cases))\n    (directory / "dataset.jsonl").write_text("".join(json.dumps(row) + "\\n" for row in sequences))\n    (directory / "recall_at_R.jsonl").write_text("".join(json.dumps(case["recall"][1]) + "\\n" for case in cases))\n    (directory / "pair_index.csv").write_text("pair_id,query_key,distractor_key\\n" + "".join(\n        f\'{case["pair_id"]},{case["query_key"]},{case["distractor_key"]}\\n\' for case in cases))\n    (directory / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\\n")\n    return manifest\n'


In [ ]:
SOURCES['runner'] = '"""Train the paper\'s restricted answer model; save every finite-run outcome.\n\nThe three acquisition updates use the ordinary loss. Continuation is a\ntractable polynomial schedule, not the existential SGD schedule in the proof.\nAdam keeps its entire moment history and uses the full pair population.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nfrom dataclasses import asdict, replace\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport platform\nimport time\n\nimport numpy as np\nimport torch\n\ntry:\n    from .core import (Config, Model, LogAdam, acquisition_rates, gradient_step,\n                       rates_at, native_gradients, boundary_log_odds, export_dataset)\nexcept ImportError:\n    from core import (Config, Model, LogAdam, acquisition_rates, gradient_step,\n                      rates_at, native_gradients, boundary_log_odds, export_dataset)\n\n\ndef clean(value):\n    if isinstance(value, dict):\n        return {str(k): clean(v) for k, v in value.items()}\n    if isinstance(value, (list, tuple)):\n        return [clean(v) for v in value]\n    if isinstance(value, torch.Tensor):\n        return clean(value.detach().cpu().tolist())\n    if isinstance(value, (float, np.floating)):\n        return float(value) if math.isfinite(value) else None\n    if isinstance(value, np.integer):\n        return int(value)\n    return value\n\n\ndef dump_json(path, value):\n    Path(path).write_text(json.dumps(clean(value), indent=2, allow_nan=False) + "\\n")\n\n\ndef write_csv(path, rows):\n    path = Path(path)\n    if not rows:\n        path.write_text("")\n        return\n    fields = list(dict.fromkeys(key for row in rows for key in row))\n    with path.open("w", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=fields)\n        writer.writeheader()\n        writer.writerows(clean(row) for row in rows)\n\n\ndef fingerprint(params):\n    digest = hashlib.sha256()\n    for value in params:\n        digest.update(value.detach().cpu().contiguous().numpy().tobytes())\n    return digest.hexdigest()\n\n\ndef _validate(c):\n    if c.n < 2 or c.d < 1 or c.R < 2 or c.steps < 3:\n        raise ValueError("Use n>=2, d>=1, integer R>=2, and steps>=3.")\n    if any(type(getattr(c, k)) is not int for k in ("n", "d", "R", "steps", "pair_batch")):\n        raise ValueError("Dimensions, lag, steps and pair batch must be integers.")\n    if not 0 <= c.beta1 < 1 or not c.beta1**2 < c.beta2 < 1:\n        raise ValueError("Adam requires 0<=beta1<1 and beta1**2<beta2<1.")\n    if not 2/3 < c.power <= 1 or c.table_power <= 1:\n        raise ValueError("Use scalar power in (2/3,1] and summable table power>1.")\n    if not .5 < c.kcal < 1 or min(c.wo, c.wc) <= 0 or not math.isclose(c.kcal+c.wo+c.wc, 1):\n        raise ValueError("Positive O/C weights and calibration>.5 must sum to one.")\n    positive = ("sigma", "gamma", "c0", "q0", "u0", "h0", "am", "ag", "ax", "aw",\n                "lr_sgd", "lr_adam", "offset", "table_lr", "eps_decay", "pair_batch")\n    if any(not math.isfinite(getattr(c, k)) or getattr(c, k) <= 0 for k in positive):\n        raise ValueError("Initialization scales and learning-rate settings must be positive and finite.")\n    if c.gamma > 1/16 or c.batch_growth < 0:\n        raise ValueError("Use acquisition gamma<=1/16 and nonnegative batch growth.")\n    if c.ag**2 <= c.c0*c.am**2:\n        raise ValueError("Binder multipliers must satisfy ag**2 > c0*am**2.")\n\n\ndef audit_native(model):\n    """Compare independent native autodiff and analytical signed-log gradients."""\n    actual, native_info = native_gradients(model)\n    signed, info = model.gradients()\n    errors = []\n    for (sign, logabs), reference in zip(signed, actual):\n        reconstructed = sign * logabs.exp()\n        denominator = max(float(reference.abs().max()), 1e-300)\n        errors.append(float((reconstructed-reference).abs().max())/denominator)\n    native_logloss = float(native_info.get("logloss", native_info.get("log_loss")))\n    return dict(relative_max_gradient_error_by_block=errors,\n                absolute_log_loss_error=abs(float(info["logloss"])-native_logloss),\n                passed=max(errors) < 2e-8 and abs(float(info["logloss"])-native_logloss) < 1e-10)\n\n\n@torch.no_grad()\ndef snapshot(model, tags, step, S, initial_tables, examples, elapsed):\n    q,p,u,v,x,w,m,g,h,rho,lrho = [float(v) for v in model.quantities()]\n    gaps = model.gaps()[model.mask]\n    delta = float(gaps.min())\n    logloss = float(model.gradients()[1]["logloss"])\n    _,_,_,so,sc = model.rows(gaps)\n    norms = max(float(model.Q.norm(dim=1).max()), float(model.K.norm(dim=1).max()))\n    movement = max(float((a-b).norm(dim=1).max()) for a,b in zip(model.params()[:2], initial_tables))\n    return dict(**tags, step=step, S=S, examples=examples, elapsed_seconds=elapsed,\n                log_loss=logloss, loss=math.exp(logloss), q=q,p=p,u=u,v=v,x=x,w=w,\n                m=m,g=g,h=h,rho=rho,log_rho=lrho,m_rho=m*rho,\n                delta_min=delta,delta_max=float(gaps.max()),row_norm_max=norms,\n                table_movement=movement, content_horizon=1+m*delta/h,\n                sg_balance=m*delta-(model.cfg.R+1)*h-math.log(m) if m>0 else None,\n                train_overwrite_min_probability=float(torch.sigmoid(w*so).min()),\n                train_recall_min_probability=float(torch.sigmoid(w*sc).min()),\n                calibration_probability=float(torch.sigmoid(model.theta[-1])),\n                certificate_valid=norms<=1 and delta>0 and w>=0 and m>=0,\n                pair_count=model.cfg.n*(model.cfg.n-1), train_R=model.cfg.R)\n\n\n@torch.no_grad()\ndef evaluate(model, tags, step, S, lags, prefixes):\n    """Exhaustive key-pair witness curves, plus an arbitrary-prefix certificate.\n\n    Probe: (a,-1)^P, (a,+1), (b,-1)^(r-1), ?a; complements have\n    exactly the same correct-answer probability. This is not all streams.\n    """\n    _,_,_,_,_,w,m,_,h,rho,_ = model.quantities()\n    gaps = model.gaps()[model.mask]\n    valid = (float(model.Q.norm(dim=1).max())<=1 and float(model.K.norm(dim=1).max())<=1\n             and float(gaps.min())>0 and float(w)>=0 and float(m)>=0)\n    rows = []\n    for lag in lags:\n        logbound = (4*m*rho + torch.logaddexp(-m*gaps.min()+(lag-1)*h, -h)\n                    -torch.log(-torch.expm1(-h)))\n        bound = float(torch.sigmoid(-w*torch.tanh(logbound/2))) if valid else None\n        for prefix in prefixes:\n            logodds = boundary_log_odds(gaps,m,h,rho,lag,prefix)\n            attention = torch.sigmoid(-logodds)\n            probability = torch.sigmoid(-w*torch.tanh(logodds/2))\n            rows.append(dict(**tags,step=step,S=S,lag=lag,prefix=prefix,\n                             mean_probability=float(probability.mean()),min_probability=float(probability.min()),\n                             mean_attention=float(attention.mean()),min_attention=float(attention.min()),\n                             certified_probability_lower_bound=bound,\n                             example_count=2*len(gaps),train_R=model.cfg.R))\n    return rows\n\n\ndef run_suite(cfg, out_dir, seeds=(0,1,2), gate_modes=("learned","retrieval_frozen"),\n              optimizers=("sgd","adam"), eval_lags=tuple(range(1,33)),\n              prefixes=(0,16,64), log_every=1000, progress=True):\n    """Return output path. Never replace an existing experiment.\n\n    `steps` includes three acquisition updates. Scalar clock S starts after\n    them. Sampling RNG and initial tensors are paired across gate variants.\n    Both optimizers start from the Gaussian draw, not an Adam-trained fork.\n    """\n    _validate(cfg)\n    if not seeds or len(set(seeds)) != len(seeds) or any(type(s) is not int or s<0 for s in seeds):\n        raise ValueError("Provide distinct nonnegative integer seeds.")\n    if not gate_modes or not set(gate_modes)<=set(("learned","retrieval_frozen","both_frozen")):\n        raise ValueError("Unknown or empty gate modes.")\n    if not optimizers or not set(optimizers)<=set(("sgd","adam")):\n        raise ValueError("Choose sgd and/or adam.")\n    if len(set(gate_modes))!=len(gate_modes) or len(set(optimizers))!=len(optimizers):\n        raise ValueError("Duplicate branches are not allowed.")\n    if not eval_lags or not prefixes or log_every<1 or any(type(x) is not int or x<1 for x in eval_lags) or any(type(x) is not int or x<0 for x in prefixes):\n        raise ValueError("Use positive integer evaluation lags/log period and nonnegative prefixes.")\n    out = Path(out_dir).resolve()\n    if out.exists() and any(out.iterdir()):\n        raise FileExistsError(f"Run directory is not empty: {out}. Choose a new run tag.")\n    out.mkdir(parents=True,exist_ok=True)\n    (out/"checkpoints").mkdir()\n    source_dir = Path(__file__).resolve().parent\n    source_bytes = {p.name:p.read_bytes() for p in source_dir.glob("*.py")}\n    hashes = {name:hashlib.sha256(data).hexdigest() for name,data in source_bytes.items()}\n    (out/"source").mkdir()\n    for name,data in source_bytes.items():\n        (out/"source"/name).write_bytes(data)\n    dump_json(out/"config.json",dict(config=asdict(cfg),seeds=seeds,gate_modes=gate_modes,\n              optimizers=optimizers,eval_lags=eval_lags,prefixes=prefixes,log_every=log_every,\n              protocol="three ordinary-answer acquisition updates; practical polynomial continuation; no moment reset",\n              numerical_backend="float64 parameters; exact signed-log analytic gradients and bias-corrected Adam buffers",\n              sgd_sampling="full population during acquisition; growing iid ordered-pair batches thereafter",\n              adam_sampling="full ordered-pair population at every update",\n              epsilon="sigma**3 * exp(-eps_decay*(global_step-1)); outside sqrt(v)",\n              source_sha256=hashes,torch_version=torch.__version__,numpy_version=np.__version__,\n              python_version=platform.python_version(),cuda_available=torch.cuda.is_available(),\n              theorem_dimension_at_failure_probability_005=math.ceil(144*math.log(16*cfg.n*(cfg.n-1)/.05)),\n              limits=["Finite schedules do not implement existential SGD continuation envelopes or R>2 basin preparation.",\n                      "Frozen answer theorem is R=2; larger R is an empirical extension.",\n                      "No holdout key pairs: this tests length/lag distribution shift on the same vocabulary.",\n                      "Finite-prefix probe minima are not minima over all possible streams."]))\n    export_dataset(cfg, out/"dataset")\n    histories,probabilities,runs,audits = [],[],[],[]\n    first_hash_by_seed = {}\n    for seed in seeds:\n        for gate_mode in gate_modes:\n            for optimizer in optimizers:\n                c = replace(cfg,seed=seed,gate_mode=gate_mode)\n                model = Model(c)\n                tags = dict(seed=seed,gate_mode=gate_mode,optimizer=optimizer)\n                label = f"seed{seed}_{gate_mode}_{optimizer}"\n                initial_hash = fingerprint(model.params())\n                first_hash_by_seed.setdefault(seed, initial_hash)\n                if initial_hash != first_hash_by_seed[seed]:\n                    raise AssertionError("Paired initial parameters differ across arms.")\n                initial_tables = [p.clone() for p in model.params()[:2]]\n                frozen_initial = model.theta.clone()\n                acquisition = acquisition_rates(c,optimizer)\n                adam = LogAdam(model.params(),c.beta1,c.beta2) if optimizer=="adam" else None\n                rng = np.random.default_rng(seed+104729)\n                start = time.monotonic()\n                S, examples, completed = 0.,0,0\n                status, reason, acquired = "finite_budget_complete", "", None\n                native = audit_native(model)\n                audits.append(dict(**tags,step=0,**native))\n                if not native["passed"]:\n                    raise AssertionError(f"Initial native gradient audit failed: {label}: {native}")\n                def record():\n                    row = snapshot(model,tags,completed,S,initial_tables,examples,time.monotonic()-start)\n                    histories.append(row)\n                    probabilities.extend(evaluate(model,tags,completed,S,eval_lags,prefixes))\n                    return row\n                record()\n                for step in range(1,c.steps+1):\n                    if time.monotonic()-start > c.max_seconds:\n                        status,reason = "time_budget_stop", "per-branch wall-clock limit"\n                        break\n                    if step<=3:\n                        base,rates,counts,batch = 0.,acquisition[step-1],None,c.n*(c.n-1)\n                    else:\n                        tail = step-4\n                        base,rates = rates_at(c,optimizer,tail)\n                        counts = None\n                        batch = c.n*(c.n-1)\n                        if optimizer=="sgd":\n                            batch = math.ceil(c.pair_batch*(1+tail/c.offset)**c.batch_growth)\n                            counts = torch.as_tensor(rng.multinomial(batch,np.full(c.n*(c.n-1),1/(c.n*(c.n-1)))),\n                                                     device=c.device,dtype=torch.float64)\n                    gradients,info = model.gradients(counts)\n                    finite = all(bool(torch.isfinite(s).all()) and bool(torch.isfinite(l[s!=0]).all())\n                                 for s,l in gradients)\n                    if not finite or not math.isfinite(float(info["logloss"])):\n                        status,reason = "numerical_stop","nonfinite signed-log loss or active gradient"\n                        break\n                    if float(info.get("table_log_span",0)) > 650:\n                        status,reason = "numerical_stop","raw-table gradient aggregation dynamic range exceeded 650 log units"\n                        break\n                    previous = [p.clone() for p in model.params()]\n                    if adam:\n                        adam.step(model.params(),gradients,rates,3*math.log(c.sigma)-c.eps_decay*(step-1))\n                    else:\n                        gradient_step(model,gradients,rates)\n                    if not all(bool(torch.isfinite(p).all()) for p in model.params()):\n                        for p,old in zip(model.params(),previous):\n                            p.copy_(old)\n                        status,reason = "numerical_stop","nonfinite parameter update; parameters restored, optimizer state invalid"\n                        break\n                    frozen_indices = getattr(model,"frozen_scalar_indices",())\n                    if frozen_indices and not torch.equal(model.theta[list(frozen_indices)],frozen_initial[list(frozen_indices)]):\n                        raise AssertionError("A frozen coordinate moved.")\n                    S += float(base)\n                    examples += 4*batch+2*c.n\n                    completed = step\n                    if step==3:\n                        acquired = bool((model.gaps()[model.mask]>0).all()) and float(model.theta[-1])>0\n                        native = audit_native(model)\n                        audits.append(dict(**tags,step=3,**native))\n                        if not native["passed"]:\n                            raise AssertionError(f"Acquired native gradient audit failed: {label}: {native}")\n                        if not acquired:\n                            status,reason = "acquisition_failed","three updates did not acquire all positive raw gaps and decoder orientation"\n                    if step==3 or step%log_every==0 or step==c.steps or status!="finite_budget_complete":\n                        row = record()\n                        if progress:\n                            print(f"{label}: {step}/{c.steps}, loss={row[\'loss\']:.3g}, min_gap={row[\'delta_min\']:.4g}, "\n                                  f"horizon={row[\'content_horizon\']:.2f}, S={S:.3g}",flush=True)\n                    if status!="finite_budget_complete":\n                        break\n                if histories[-1]["step"] != completed or any(histories[-1][k]!=v for k,v in tags.items()):\n                    record()\n                run = dict(**tags,completed_steps=completed,requested_steps=c.steps,status=status,reason=reason,\n                           initial_hash=initial_hash,final_hash=fingerprint(model.params()),\n                           acquired_positive_gaps=acquired,train_R=c.R,S=S)\n                runs.append(run)\n                checkpoint = dict(config=asdict(c),model=[p.cpu() for p in model.params()],\n                                  optimizer=None if adam is None else dict(t=adam.t,b1=adam.b1,b2=adam.b2,\n                                       ms=[p.cpu() for p in adam.ms],ml=[p.cpu() for p in adam.ml],vl=[p.cpu() for p in adam.vl]),\n                                  rng_state=rng.bit_generator.state,summary=run,\n                                  optimizer_state_valid="restored" not in reason)\n                torch.save(checkpoint,out/"checkpoints"/(label+".pt"))\n                write_csv(out/"history.csv",histories)\n                write_csv(out/"lag_probabilities.csv",probabilities)\n                write_csv(out/"runs.csv",runs)\n                dump_json(out/"native_gradient_audits.json",audits)\n                if progress:\n                    print(f"{label}: {status}"+(f" ({reason})" if reason else ""),flush=True)\n    return str(out)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--out",required=True)\n    parser.add_argument("--n",type=int,default=8)\n    parser.add_argument("--R",type=int,default=2)\n    parser.add_argument("--d",type=int,default=2048)\n    parser.add_argument("--steps",type=int,default=12000)\n    parser.add_argument("--device",default="cpu")\n    parser.add_argument("--seeds",type=int,nargs="+",default=[0,1,2])\n    parser.add_argument("--max-lag",type=int,default=32)\n    parser.add_argument("--log-every",type=int,default=1000)\n    args = parser.parse_args()\n    if args.device=="cpu":\n        torch.set_num_threads(1)\n    cfg = Config(n=args.n,R=args.R,d=args.d,steps=args.steps,device=args.device)\n    path = run_suite(cfg,args.out,seeds=tuple(args.seeds),eval_lags=tuple(range(1,args.max_lag+1)),log_every=args.log_every)\n    try:\n        from .report import make_report\n    except ImportError:\n        from report import make_report\n    make_report(path)\n\n\nif __name__=="__main__":\n    main()\n'


In [ ]:
SOURCES['report'] = '"""Plot the measured restricted-model experiment without importing the trainer.\n\nOnly numpy and matplotlib are required.  CSV files remain the source of truth;\nfigures never extrapolate an optimizer trajectory beyond its saved checkpoints.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport json\nimport math\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport matplotlib\n\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\n\n\nCOLORS = {"sgd": "#2166ac", "adam": "#d95f02"}\nGATE_NAMES = {\n    "learned": "Learned retrieval gate",\n    "retrieval_frozen": "Frozen retrieval gate (ALiBi)",\n    "both_frozen": "Both gates frozen (extra control)",\n}\nKEY_FIELDS = ("seed", "gate_mode", "optimizer")\n\n\ndef _number(value, default=float("nan")):\n    try:\n        return float(value)\n    except (TypeError, ValueError):\n        return default\n\n\ndef _read_csv(path):\n    if not path.exists() or not path.stat().st_size:\n        return []\n    with path.open(newline="", encoding="utf-8") as handle:\n        return list(csv.DictReader(handle))\n\n\ndef _key(row):\n    return tuple(str(row.get(field, "")) for field in KEY_FIELDS)\n\n\ndef _group(rows):\n    result = defaultdict(list)\n    for row in rows:\n        result[_key(row)].append(row)\n    return result\n\n\ndef _latest(rows):\n    """Select the last evaluated checkpoint separately for every branch."""\n    result = []\n    for branch in _group(rows).values():\n        step = max((_number(row.get("step"), -1) for row in branch), default=-1)\n        result.extend(row for row in branch if _number(row.get("step"), -1) == step)\n    return result\n\n\ndef _mean(values):\n    values = [v for v in values if math.isfinite(v)]\n    return float(np.mean(values)) if values else float("nan")\n\n\ndef _minimum(values):\n    values = [v for v in values if math.isfinite(v)]\n    return min(values) if values else float("nan")\n\n\ndef _per_seed_lag(rows, metric="min_probability"):\n    # Each CSV row already aggregates ordered key pairs for one prefix. Taking\n    # a second minimum therefore gives a minimum over the finite test panel.\n    groups = defaultdict(list)\n    for row in rows:\n        groups[(_key(row), _number(row.get("lag")))].append(_number(row.get(metric)))\n    result = defaultdict(dict)\n    for (branch, lag), values in groups.items():\n        if math.isfinite(lag):\n            result[branch][lag] = _minimum(values)\n    return result\n\n\ndef _per_seed_mean_lag(rows):\n    groups = defaultdict(list)\n    for row in rows:\n        value = _number(row.get("mean_probability"))\n        weight = _number(row.get("example_count"), 1)\n        lag = _number(row.get("lag"))\n        if math.isfinite(value) and math.isfinite(weight) and weight > 0 and math.isfinite(lag):\n            groups[(_key(row), lag)].append((value, weight))\n    result = defaultdict(dict)\n    for (key, lag), values in groups.items():\n        result[key][lag] = sum(v * w for v, w in values) / sum(w for _, w in values)\n    return result\n\n\ndef _format(value, digits=4):\n    value = _number(value)\n    return f"{value:.{digits}g}" if math.isfinite(value) else "unavailable"\n\n\ndef _save(fig, path, pdf=False):\n    fig.savefig(path, dpi=180, bbox_inches="tight", facecolor="white")\n    if pdf:\n        fig.savefig(path.with_suffix(".pdf"), bbox_inches="tight", facecolor="white")\n    plt.close(fig)\n\n\ndef _empty(ax, message="No saved observations"):\n    ax.text(.5, .5, message, ha="center", va="center", transform=ax.transAxes, color="0.4")\n\n\ndef _finish_axes(axes):\n    for ax in np.asarray(axes).ravel():\n        ax.grid(alpha=.17)\n        ax.spines[["top", "right"]].set_visible(False)\n        handles, labels = ax.get_legend_handles_labels()\n        if handles:\n            unique = dict(zip(labels, handles))\n            ax.legend(unique.values(), unique.keys(), fontsize=8, frameon=False,\n                      ncol=2 if len(unique) > 6 else 1)\n\n\ndef _lag_plot(final_rows, gates, R, path):\n    fig, axes = plt.subplots(1, len(gates), figsize=(6.3 * len(gates), 4.6), squeeze=False)\n    series = _per_seed_lag(final_rows)\n    mean_series = _per_seed_mean_lag(final_rows)\n    for ax, gate in zip(axes.ravel(), gates):\n        has_data = False\n        for opt, color in COLORS.items():\n            branches = [vals for key, vals in series.items() if key[1:] == (gate, opt)]\n            if not branches:\n                continue\n            has_data = True\n            for values in branches:\n                xs = sorted(values)\n                ax.plot(xs, [values[x] for x in xs], color=color, alpha=.18, lw=.9)\n            xs = sorted({lag for values in branches for lag in values})\n            # Unequal finite run budgets remain explicit in report.md. This\n            # envelope is a seed range, not a confidence interval.\n            batches = [[v[x] for v in branches if x in v and math.isfinite(v[x])] for x in xs]\n            means = [_mean(batch) for batch in batches]\n            lower = [_minimum(batch) for batch in batches]\n            upper = [max(batch) if batch else float("nan") for batch in batches]\n            ax.fill_between(xs, lower, upper, color=color, alpha=.12)\n            ax.plot(xs, means, color=color, lw=2.2, label=f"{opt.upper()}: seed mean of minima")\n            mean_branches = [vals for key, vals in mean_series.items() if key[1:] == (gate, opt)]\n            if mean_branches:\n                ax.plot(xs, [_mean([vals[x] for vals in mean_branches if x in vals]) for x in xs],\n                        color=color, ls="--", lw=1.3, label=f"{opt.upper()}: mean probability")\n        for threshold in (.5, .9):\n            ax.axhline(threshold, color=".55", linestyle=":", lw=.8)\n            ax.text(.985, threshold + .01, f"{threshold:g}", transform=ax.get_yaxis_transform(),\n                    ha="right", fontsize=8, color=".4")\n        if R is not None:\n            ax.axvline(R, color=".3", linestyle="--", lw=1, label=f"Training lag R = {R}")\n            if gate == "learned" and R >= 2:\n                ax.axvline(R + 2, color="#6a51a3", linestyle=":", lw=1.4,\n                           label=("SGD horizon 4 (asymptotic theorem)" if R == 2\n                                  else "SGD horizon R + 2 (conditional asymptotic)"))\n        if not has_data:\n            _empty(ax)\n        ax.set(title=GATE_NAMES.get(gate, gate), xlabel="Target lag r (records)",\n               ylabel="Correct-answer probability", ylim=(-.025, 1.045))\n    fig.suptitle("Generalization at the latest saved checkpoint of each run", fontsize=14)\n    fig.text(.5, .015, "Solid: seed mean of finite-panel minima. Dashed: mean over tested examples, then seeds. "\n             "Band: range of seed minima. Finite runs do not establish asymptotic limits.", ha="center", fontsize=8)\n    _finish_axes(axes)\n    fig.tight_layout(rect=(0, .06, 1, .94))\n    _save(fig, path, pdf=True)\n\n\ndef _training_plot(rows, history, gates, R, path):\n    fig, axes = plt.subplots(len(gates), 2, figsize=(12.7, 3.65 * len(gates)), squeeze=False)\n    history_lookup = {(_key(row), _number(row.get("step"))): row for row in history}\n    chosen_lags = [R, R + 2, R + 3, 2 * R + 4] if R is not None else []\n    if not chosen_lags:\n        chosen_lags = sorted({_number(row.get("lag")) for row in rows if math.isfinite(_number(row.get("lag")))})[:4]\n    chosen_lags = list(dict.fromkeys(chosen_lags))\n    lag_colors = ["#1b9e77", "#7570b3", "#d95f02", "#e7298a"]\n    aggregated = defaultdict(list)\n    for row in rows:\n        aggregated[(_key(row), _number(row.get("step")), _number(row.get("lag")))].append(row)\n    curves = defaultdict(list)\n    for (branch, step, lag), batch in aggregated.items():\n        source = history_lookup.get((branch, step), {})\n        S = _number(batch[0].get("S"))\n        if not math.isfinite(S):\n            S = _number(source.get("S"))\n        value = _minimum([_number(row.get("min_probability")) for row in batch])\n        if math.isfinite(S) and math.isfinite(value):\n            curves[(branch, lag)].append((S, step, value))\n    for row_index, gate in enumerate(gates):\n        for col_index, opt in enumerate(COLORS):\n            ax = axes[row_index, col_index]\n            has_data = False\n            seed_styles = {}\n            for lag, color in zip(chosen_lags, lag_colors):\n                branches = [(key, values) for (key, r), values in curves.items() if key[1:] == (gate, opt) and r == lag]\n                for branch_index, (key, values) in enumerate(sorted(branches)):\n                    values = sorted(values, key=lambda point: point[1])\n                    linestyle = ("-", "--", "-.", ":")[branch_index % 4]\n                    seed_styles[key[0]] = linestyle\n                    ax.plot([p[0] for p in values], [p[2] for p in values], color=color,\n                            lw=1.5, alpha=.8, ls=linestyle,\n                            label=f"r = {lag:g}" if branch_index == 0 else "_nolegend_")\n                    has_data = True\n            for seed, linestyle in sorted(seed_styles.items()):\n                ax.plot([], [], color=".4", ls=linestyle, lw=1.2, label=f"seed {seed}")\n            ax.axhline(.5, color=".55", ls=":", lw=.8)\n            ax.axhline(.9, color=".55", ls=":", lw=.8)\n            ax.set_xscale("symlog", linthresh=1)\n            ax.set(title=f"{GATE_NAMES.get(gate, gate)} · {opt.upper()}",\n                   xlabel="Continuation learning-rate sum S (own run schedule)",\n                   ylabel="Minimum correct-answer probability", ylim=(-.025, 1.045))\n            if not has_data:\n                _empty(ax, "No saved probabilities at the selected lags")\n    fig.suptitle("Probability during training at selected test lags", fontsize=14)\n    fig.text(.5, .012, "Each seed keeps its own learning-rate sum; curves are not aligned or averaged across different S values.",\n             ha="center", fontsize=9)\n    _finish_axes(axes)\n    fig.tight_layout(rect=(0, .045, 1, .95))\n    _save(fig, path)\n\n\ndef _diagnostic_plot(history, final_rows, gates, R, path):\n    fig, axes = plt.subplots(len(gates), 4, figsize=(19, 3.9 * len(gates)), squeeze=False)\n    specifications = [\n        ("log_loss", "Log ordinary answer loss", "ln(loss)"),\n        ("content_horizon", "Content horizon", "1 + m δ_min / h"),\n        ("m_rho", "Binding-contamination scale", "m ρ"),\n    ]\n    for gate_index, gate in enumerate(gates):\n        for metric_index, (metric, title, ylabel) in enumerate(specifications):\n            ax = axes[gate_index, metric_index]\n            has_data = False\n            for key, branch in sorted(_group(history).items()):\n                if key[1] != gate:\n                    continue\n                points = []\n                for row in sorted(branch, key=lambda row: _number(row.get("step"), -1)):\n                    x, y = _number(row.get("S")), _number(row.get(metric))\n                    if metric == "log_loss" and not math.isfinite(y):\n                        loss = _number(row.get("loss"))\n                        y = math.log(loss) if loss > 0 else float("nan")\n                    if math.isfinite(x) and math.isfinite(y):\n                        points.append((x, y))\n                if points:\n                    ax.plot(*zip(*points), color=COLORS.get(key[2], ".3"), lw=1.5,\n                            alpha=.8, label=f"{key[2].upper()}, seed {key[0]}")\n                    has_data = True\n            ax.set_xscale("symlog", linthresh=1)\n            if metric == "m_rho":\n                ax.set_yscale("symlog", linthresh=1e-7)\n            if metric == "content_horizon" and R is not None:\n                ax.axhline(R, color=".5", ls="--", lw=.8, label="Training lag R")\n                if gate == "learned" and R >= 2:\n                    ax.axhline(R + 2, color="#6a51a3", ls=":", lw=1,\n                               label=("SGD asymptote 4 (R = 2)" if R == 2\n                                      else "SGD R + 2 (conditional, R > 2)"))\n            ax.set(title=f"{GATE_NAMES.get(gate, gate)}\\n{title}", xlabel="Continuation sum S", ylabel=ylabel)\n            if not has_data:\n                _empty(ax)\n        ax = axes[gate_index, 3]\n        for metric, label, linestyle in [("min_probability", "answer", "-"), ("min_attention", "attention", "--")]:\n            series = _per_seed_lag(final_rows, metric)\n            for opt, color in COLORS.items():\n                branches = [vals for key, vals in series.items() if key[1:] == (gate, opt)]\n                xs = sorted({lag for values in branches for lag in values})\n                if xs:\n                    ax.plot(xs, [_mean([v[x] for v in branches if x in v]) for x in xs],\n                            color=color, ls=linestyle, lw=1.8, label=f"{opt.upper()} {label}")\n        ax.axhline(.5, color=".55", ls=":", lw=.8)\n        ax.set(title="Latest checkpoint: attention vs answer", xlabel="Target lag r (records)",\n               ylabel="Seed mean of finite-panel minima", ylim=(-.025, 1.045))\n        if not final_rows:\n            _empty(ax)\n    fig.suptitle("Mechanism diagnostics from saved checkpoints", fontsize=15)\n    fig.text(.5, .012, "A large content horizon is a diagnostic, not a probability certificate; "\n             "answer confidence also depends on binding, old prefixes, and output scale.", ha="center", fontsize=9)\n    _finish_axes(axes)\n    fig.tight_layout(rect=(0, .045, 1, .95))\n    _save(fig, path)\n\n\ndef _markdown(config_data, history, lag_rows, runs, R):\n    config = config_data.get("config", config_data)\n    final_rows = _latest(lag_rows)\n    final_history = {_key(row): row for row in _latest(history)}\n    settings = []\n    for name in ("n", "d", "R", "steps"):\n        if name in config:\n            settings.append(f"{name}={config[name]}")\n    if "R" not in config and R is not None:\n        settings.append(f"R={R}")\n    lines = ["# Restricted-model empirical report", "",\n             "These results describe the saved finite training runs. They can test consistency with the theory; "\n             "they do not prove the asymptotic optimizer claims.", ""]\n    if settings:\n        lines.extend(["Settings: " + ", ".join(settings) + ". See `config.json` for initialization, optimizer, and evaluation settings.", ""])\n    prefixes = sorted({_number(row.get("prefix")) for row in lag_rows if math.isfinite(_number(row.get("prefix")))})\n    lags = sorted({_number(row.get("lag")) for row in lag_rows if math.isfinite(_number(row.get("lag")))})\n    lines.extend([\n        "- Checkpoint step 0 is the paired Gaussian initialization; step 3 ends optimizer-specific acquisition. "\n        "Continuation uses each optimizer\'s own learning-rate sum S.",\n        "- The frozen-retrieval comparison freezes only the retrieval forgetting parameter h. "\n        "The binding gate remains trainable. Any `both_frozen` arm is a separate extra control.",\n        "- The SGD continuation schedule is a practical experiment schedule, not the literal conservative theorem schedule.",\n        "- Frozen-gate arbitrary-prefix success requires the theorem\'s conditions, including h₀ > log(2). "\n        "Positive matching gaps, suitable row bounds, controlled binding contamination, and positive output scale must also hold.",\n        "- Learned-gate SGD\'s asymptotic horizon is 4 in the main R = 2 theorem. "\n        "The R + 2 reference for R > 2 is conditional on the generalized theorem\'s hypotheses. "\n        "At the boundary, output probability must be checked separately from target attention.",\n        "- The fully proved frozen-retrieval answer result uses R=2. Frozen R>2 and learned-Adam R>2 "\n        "runs are empirical extension tests here; the notebook also omits the additional R>2 SGD basin preparation.",\n        "- Test prefix lengths: " + (", ".join(f"{p:g}" for p in prefixes) or "none saved") + ". "\n        "Test lags: " + (", ".join(f"{r:g}" for r in lags) or "none saved") + ".", "",\n        "## Run completion and latest observations", "",\n        "Plots use the latest saved evaluation separately for each branch. A stopped run may therefore have an earlier "\n        "checkpoint than a completed run. Seed envelopes are observed ranges, not confidence intervals.", "",\n        "| Seed | Gate | Optimizer | Completed / requested | Status | Evaluation step | Reason |",\n        "| --- | --- | --- | --- | --- | --- | --- |",\n    ])\n    final_steps = {_key(row): _format(row.get("step"), 8) for row in final_rows}\n    branch_runs = list(runs)\n    known = {_key(row) for row in branch_runs}\n    for key, row in final_history.items():\n        if key not in known:\n            branch_runs.append(dict(row, completed_steps=row.get("step", ""), requested_steps="unavailable"))\n    for row in sorted(branch_runs, key=_key):\n        key = _key(row)\n        reason = str(row.get("reason", "")).replace("|", "/").replace("\\n", " ") or "—"\n        lines.append(f"| {key[0]} | {key[1]} | {key[2]} | {row.get(\'completed_steps\', \'unavailable\')} / "\n                     f"{row.get(\'requested_steps\', \'unavailable\')} | {row.get(\'status\', \'unavailable\')} | "\n                     f"{final_steps.get(key, \'none\')} | {reason} |")\n    if not branch_runs:\n        lines.append("| — | — | — | — | No saved runs | — | — |")\n    lines.extend(["", "## Generalization at selected lags", "",\n                  "For each seed, the observed minimum is over all tested ordered key pairs and prefix lengths. "\n                  "The table gives the mean and worst of those per-seed minima. It does not cover untested prefixes. "\n                  "An analytic certificate, when present, is a separate conditional arbitrary-prefix lower bound; "\n                  "blank or unavailable certificates are not empirical failures.", "",\n                  "| Gate | Optimizer | Lag | Seeds | Mean probability | Mean of observed minima | Worst observed minimum | Analytic lower bound (worst seed) |",\n                  "| --- | --- | --- | --- | --- | --- | --- | --- |"])\n    selected = set([R, R + 2, R + 3, 2 * R + 4]) if R is not None else set(lags[:4])\n    series = _per_seed_lag(final_rows)\n    mean_series = _per_seed_mean_lag(final_rows)\n    certificates = _per_seed_lag(final_rows, "certified_probability_lower_bound")\n    groups = defaultdict(list)\n    for key, values in series.items():\n        for lag, value in values.items():\n            if lag in selected:\n                groups[(key[1], key[2], lag)].append((key, value))\n    for (gate, opt, lag), observations in sorted(groups.items()):\n        values = [value for _, value in observations]\n        bounds = [certificates.get(key, {}).get(lag, float("nan")) for key, _ in observations]\n        mean_probability = _mean([mean_series.get(key, {}).get(lag, float("nan")) for key, _ in observations])\n        # Never label a partial collection as a certificate across every seed.\n        bound = min(bounds) if bounds and all(math.isfinite(v) for v in bounds) else float("nan")\n        lines.append(f"| {gate} | {opt} | {lag:g} | {len(values)} | {_format(mean_probability)} | {_format(_mean(values))} | "\n                     f"{_format(_minimum(values))} | {_format(bound)} |")\n    if not groups:\n        lines.append("| — | — | — | 0 | unavailable | unavailable | unavailable | unavailable |")\n    lines.extend(["", "## Final mechanism diagnostics", "",\n                  "| Seed | Gate | Optimizer | Step | ln(loss) | δ_min | Max row norm | h | mρ | Content horizon | SGD balance |",\n                  "| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |"])\n    for key, row in sorted(final_history.items()):\n        metrics = [row.get(name) for name in ("step", "log_loss", "delta_min", "row_norm_max", "h", "m_rho", "content_horizon", "sg_balance")]\n        lines.append("| " + " | ".join([*key, *[_format(v) for v in metrics]]) + " |")\n    if not final_history:\n        lines.append("| — | — | — | — | unavailable | unavailable | unavailable | unavailable | unavailable | unavailable | unavailable |")\n    lines.extend(["", "The content horizon is 1 + m δ_min / h; the SGD balance is m δ_min − (R + 1)h − log(m). "\n                  "These are mechanism diagnostics, not substitutes for the actual answer probabilities. "\n                  "`theory_diagnostics.png` displays target attention separately from correct-answer probability, "\n                  "since the two can behave differently.", "",\n                  "## Saved figures", "",\n                  "- `generalization_by_lag.png` and `.pdf`: probability versus lag at each branch\'s latest evaluation.",\n                  "- `probability_by_training.png`: selected-lag probabilities along each saved training trajectory.",\n                  "- `theory_diagnostics.png`: log loss, content horizon, binding contamination, and attention versus answer probability.",\n                  "", "The CSV files contain all measurements and stopped-run statuses. Unobserved checkpoints are never filled in.", ""])\n    return "\\n".join(lines)\n\n\ndef make_report(out_dir):\n    """Read an experiment directory and return the generated artifact paths.\n\n    Empty or stopped experiments still produce explicit, readable placeholder\n    figures and a status report. Nothing is imputed as a successful outcome.\n    """\n    out_dir = Path(out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    history = _read_csv(out_dir / "history.csv")\n    lag_rows = _read_csv(out_dir / "lag_probabilities.csv")\n    runs = _read_csv(out_dir / "runs.csv")\n    config_path = out_dir / "config.json"\n    config_data = json.loads(config_path.read_text(encoding="utf-8")) if config_path.exists() else {}\n    config = config_data.get("config", config_data)\n    R = _number(config.get("R", config.get("train_R")))\n    if not math.isfinite(R):\n        R = next((_number(row.get("train_R")) for row in history + runs if math.isfinite(_number(row.get("train_R")))), float("nan"))\n    R = int(R) if math.isfinite(R) else None\n    observed_gates = {str(row.get("gate_mode")) for row in history + lag_rows + runs if row.get("gate_mode")}\n    observed_gates.update(config_data.get("gate_modes", []))\n    gates = [gate for gate in GATE_NAMES if gate in observed_gates]\n    gates += sorted(observed_gates - set(gates))\n    gates = gates or ["learned", "retrieval_frozen"]\n    paths = {\n        "generalization_by_lag": str(out_dir / "generalization_by_lag.png"),\n        "generalization_by_lag_pdf": str(out_dir / "generalization_by_lag.pdf"),\n        "probability_by_training": str(out_dir / "probability_by_training.png"),\n        "theory_diagnostics": str(out_dir / "theory_diagnostics.png"),\n        "report": str(out_dir / "report.md"),\n    }\n    with plt.rc_context({"font.family": "DejaVu Sans", "font.size": 10, "axes.titlesize": 11}):\n        _lag_plot(_latest(lag_rows), gates, R, Path(paths["generalization_by_lag"]))\n        _training_plot(lag_rows, history, gates, R, Path(paths["probability_by_training"]))\n        _diagnostic_plot(history, _latest(lag_rows), gates, R, Path(paths["theory_diagnostics"]))\n    Path(paths["report"]).write_text(_markdown(config_data, history, lag_rows, runs, R), encoding="utf-8")\n    return paths\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("out_dir", type=Path, help="Directory containing experiment CSV files")\n    args = parser.parse_args()\n    for name, path in make_report(args.out_dir).items():\n        print(f"{name}: {path}")\n'


In [ ]:
SOURCES['test_core'] = '"""Independent stream, derivative, optimizer, initialization, and dataset QA."""\nimport json\nimport math\nfrom pathlib import Path\nimport tempfile\nimport unittest\n\nimport torch\n\ntry:\n    from .core import (Config, Model, LogAdam, acquisition_rates, boundary_log_odds,\n                       dense_objective, dense_stream, export_dataset, gradient_step,\n                       native_gradients, native_loss, pair_cases, rates_at, slog)\nexcept ImportError:\n    from core import (Config, Model, LogAdam, acquisition_rates, boundary_log_odds,\n                      dense_objective, dense_stream, export_dataset, gradient_step,\n                      native_gradients, native_loss, pair_cases, rates_at, slog)\n\n\ndef moderate_model(R=2, gate_mode="learned", w=.8):\n    c = Config(n=3, d=5, R=R, gate_mode=gate_mode, D=.3, c0=1.2)\n    model = Model(c)\n    gen = torch.Generator().manual_seed(812)\n    model.Q = torch.randn(c.n, c.d, generator=gen, dtype=torch.float64) * .3\n    model.K = torch.randn(c.n, c.d, generator=gen, dtype=torch.float64) * .3\n    model.theta = torch.tensor([.8, .9, .65, .8, .4, w], dtype=torch.float64)\n    return model\n\n\nclass CoreTests(unittest.TestCase):\n    @classmethod\n    def setUpClass(cls):\n        torch.set_num_threads(1)\n\n    def test_exact_stream_loss_and_gradients(self):\n        for R in (2, 3, 4, 7):\n            for gate_mode in ("learned", "retrieval_frozen", "both_frozen"):\n                for w in (-.6, 0., .8):\n                    for sampled in (False, True):\n                        with self.subTest(R=R, gate=gate_mode, w=w, sampled=sampled):\n                            model = moderate_model(R, gate_mode, w)\n                            counts = torch.tensor([0., 3., 1., 2., 0., 4.]) if sampled else None\n                            params = [p.clone().requires_grad_(True) for p in model.params()]\n                            loss_dense = dense_objective(model, counts, params)\n                            expected = torch.autograd.grad(loss_dense, params)\n                            native, info = native_gradients(model, counts)\n                            signed, slog_info = model.gradients(counts)\n                            torch.testing.assert_close(info["loss"], loss_dense, rtol=2e-12, atol=2e-14)\n                            torch.testing.assert_close(slog_info["logloss"].exp(), loss_dense, rtol=2e-12, atol=2e-14)\n                            for target, actual, (sign, logabs) in zip(expected, native, signed):\n                                self.assertFalse(bool(torch.isnan(logabs).any()))\n                                torch.testing.assert_close(actual, target, rtol=2e-10, atol=2e-13)\n                                torch.testing.assert_close(sign * logabs.exp(), target, rtol=2e-10, atol=2e-13)\n\n    def test_boundary_odds_match_independent_attention(self):\n        for gate_mode in ("learned", "retrieval_frozen", "both_frozen"):\n            model = moderate_model(4, gate_mode)\n            q, p, u, v, x, w, m, g, h, rho, lrho = model.quantities()\n            for lag in (1, 2, 3, 6, 20):\n                for prefix in (0, 1, 9):\n                    for a, b in ((0, 1), (1, 2)):\n                        keys = [a] * (prefix + 1) + [b] * (lag - 1)\n                        values = [-1] * prefix + [1] + [-1] * (lag - 1)\n                        loss, mass, logit = dense_stream(model.params(), model.cfg, keys, values, a)\n                        lo = boundary_log_odds(model.gaps()[a, b], m, h, rho, lag, prefix)\n                        torch.testing.assert_close(mass[prefix], torch.sigmoid(-lo), rtol=2e-12, atol=2e-14)\n                        torch.testing.assert_close(logit, -w * torch.tanh(lo / 2), rtol=2e-12, atol=2e-14)\n\n    def test_trusted_counts_path_matches_checked_path(self):\n        model = moderate_model(4)\n        counts = torch.tensor([0., 3., 1., 2., 0., 4.], dtype=torch.float64)\n        checked, checked_info = model.gradients(counts)\n        trusted, trusted_info = model.gradients(counts, validate_counts=False)\n        for expected, actual in zip(checked, trusted):\n            for x, y in zip(expected, actual):\n                torch.testing.assert_close(x, y, rtol=0, atol=0)\n        torch.testing.assert_close(checked_info["table_log_span"], trusted_info["table_log_span"])\n\n    def test_log_adam_matches_torch_adam_complete_history(self):\n        for b1, b2 in ((.9, .999), (0., .9)):\n            left = [torch.tensor([.2, -.7, .4], dtype=torch.float64)]\n            right = [left[0].clone().requires_grad_(True)]\n            log_adam = LogAdam(left, b1, b2)\n            reference = torch.optim.Adam(right, lr=.01, betas=(b1, b2), eps=1e-8)\n            for t in range(30):\n                grad = torch.tensor([math.sin(t), math.cos(t) * .001, 0.], dtype=torch.float64)\n                right[0].grad = grad.clone()\n                reference.step()\n                log_adam.step(left, [slog(grad)], [.01], math.log(1e-8))\n                torch.testing.assert_close(left[0], right[0], rtol=2e-12, atol=3e-14)\n            self.assertEqual(log_adam.t, 30)\n\n    def test_annealed_adam_matches_torch(self):\n        left = [torch.tensor([.1, .3], dtype=torch.float64)]\n        right = [left[0].clone().requires_grad_(True)]\n        optimizer = LogAdam(left, .9, .999)\n        ref = torch.optim.Adam(right, lr=.02, betas=(.9, .999), eps=.001)\n        for t in range(20):\n            eps = .001 * math.exp(-.3 * t)\n            grad = torch.tensor([(-1.) ** t * .01, math.exp(-t)], dtype=torch.float64)\n            ref.param_groups[0]["eps"] = eps\n            right[0].grad = grad\n            ref.step()\n            optimizer.step(left, [slog(grad)], [.02], math.log(eps))\n        torch.testing.assert_close(left[0], right[0], rtol=2e-12, atol=3e-14)\n\n    def test_frozen_coordinates_and_buffers(self):\n        for gate_mode in ("retrieval_frozen", "both_frozen"):\n            for kind in ("sgd", "adam"):\n                model = moderate_model(4, gate_mode)\n                before = model.theta.clone()\n                optimizer = LogAdam(model.params(), .9, .999)\n                frozen = list(model.frozen_scalar_indices)\n                for _ in range(5):\n                    gradients, _ = model.gradients()\n                    self.assertTrue(bool((gradients[-1][0][frozen] == 0).all()))\n                    rates = [.001, .001, .01]\n                    if kind == "adam":\n                        optimizer.step(model.params(), gradients, rates, math.log(1e-8))\n                    else:\n                        gradient_step(model, gradients, rates)\n                torch.testing.assert_close(before[frozen], model.theta[frozen], rtol=0, atol=0)\n                if kind == "adam":\n                    self.assertTrue(bool(torch.isneginf(optimizer.vl[-1][frozen]).all()))\n                    self.assertTrue(bool((optimizer.ms[-1][frozen] == 0).all()))\n                _, rates = rates_at(model.cfg, kind, 12)\n                self.assertTrue(bool((rates[-1][frozen] == 0).all()))\n\n    def test_initialization_and_R_adjusted_acquisition(self):\n        for R in (2, 4, 7):\n            c = Config(n=4, d=256, R=R)\n            model = Model(c)\n            self.assertFalse(torch.equal(model.Q, model.K))\n            self.assertEqual(float(model.theta[-1]), 0.)\n            self.assertAlmostEqual(float(model.quantities()[8]), c.h0)\n            self.assertEqual(float(model.theta[0]), float(model.theta[1]))\n            self.assertEqual(float(model.theta[2]), float(model.theta[3]))\n            gradients, _ = model.gradients()\n            self.assertTrue(bool((gradients[0][0] == 0).all()))\n            self.assertTrue(bool((gradients[1][0] == 0).all()))\n            self.assertTrue(bool((gradients[-1][0][:-1] == 0).all()))\n            rates = acquisition_rates(c, "sgd")\n            probe = Model(c)\n            probe.Q.zero_(); probe.K.zero_(); probe.theta[-1] = .2\n            z = torch.zeros((), dtype=torch.float64, requires_grad=True)\n            _, _, _, so, sc = probe.rows(z)\n            phi = c.wo * torch.nn.functional.softplus(-.2 * so) + c.wc * torch.nn.functional.softplus(-.2 * sc)\n            astar = -float(torch.autograd.grad(phi, z)[0]) / (c.n - 1)\n            self.assertAlmostEqual(rates[2][0], 1 / astar)\n            for rate in rates:\n                grads, _ = model.gradients()\n                gradient_step(model, grads, rate)\n            self.assertGreater(float(model.gaps()[model.mask].min()), 0.)\n            for actual, (sign, logabs) in zip(native_gradients(model)[0], model.gradients()[0]):\n                torch.testing.assert_close(actual, sign * logabs.exp(), rtol=2e-9, atol=1e-13)\n\n    def test_dataset_pair_count_and_weighted_loss(self):\n        model = moderate_model(5)\n        cases = pair_cases(model.cfg)\n        self.assertEqual(len(cases), model.cfg.n * (model.cfg.n - 1))\n        for case in cases:\n            self.assertNotEqual(case["query_key"], case["distractor_key"])\n            self.assertEqual({row["target_lag"] for row in case["recall"]}, {5})\n            self.assertEqual({row["target_lag"] for row in case["overwrite"]}, {1})\n        with tempfile.TemporaryDirectory() as path:\n            manifest = export_dataset(model.cfg, path)\n            expanded = [json.loads(line) for line in (Path(path) / "dataset.jsonl").read_text().splitlines()]\n            recall = (Path(path) / "recall_at_R.jsonl").read_text().splitlines()\n            self.assertEqual(len(recall), len(cases))\n            self.assertEqual(len(expanded), 4 * len(cases) + 2 * model.cfg.n)\n            self.assertAlmostEqual(sum(row["population_weight"] for row in expanded), 1.)\n            loss = sum(row["population_weight"] * dense_stream(model.params(), model.cfg, row["keys"],\n                                                               row["values"], row["query"], row["answer"])[0]\n                       for row in expanded)\n            torch.testing.assert_close(loss, native_loss(model), rtol=2e-12, atol=2e-14)\n\n    def test_log_backend_retains_tiny_binder_gradients(self):\n        model = moderate_model(4)\n        model.theta[2:4] = 30.\n        log_gradients, _ = model.gradients()\n        native, _ = native_gradients(model)\n        self.assertEqual(float(native[-1][2]), 0.)\n        self.assertTrue(bool(torch.isfinite(log_gradients[-1][1][2])))\n        self.assertLess(float(log_gradients[-1][1][2]), -1000.)\n        self.assertNotEqual(float(log_gradients[-1][0][2]), 0.)\n\n    def test_frozen_stale_tail_threshold(self):\n        for h in (.5, math.log(2), 1.):\n            model = Model(Config(n=3, d=5, gate_mode="retrieval_frozen", h0=h))\n            q, p, u, v, x, w, m, g, h, rho, lrho = model.quantities()\n            lo = boundary_log_odds(model.gaps()[model.mask], m, h, rho, 1, math.inf)\n            torch.testing.assert_close(torch.sigmoid(-lo), torch.full_like(lo, 1 - math.exp(-float(h))))\n            if float(h) < math.log(2):\n                self.assertTrue(bool((lo > 0).all()))\n\n    def test_config_and_counts_reject_invalid_inputs(self):\n        with self.assertRaises(ValueError):\n            Config(beta1=.99, beta2=.9)\n        with self.assertRaises(ValueError):\n            Config(R=1)\n        with self.assertRaises(ValueError):\n            Config(h0=1., x0=-1.)\n        with self.assertRaises(ValueError):\n            Config(kcal=.4, wo=.3, wc=.3)\n        model = moderate_model()\n        for counts in (torch.zeros(6), torch.ones(3), -torch.ones(6)):\n            with self.assertRaises(ValueError):\n                model.gradients(counts)\n\n\nif __name__ == "__main__":\n    unittest.main(verbosity=2)\n'


In [ ]:
SOURCES['test_runner'] = '"""Integration checks for paired training, lag evaluation and saved artifacts."""\nimport csv\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pytest\nimport torch\n\nfrom core import Config, Model\nfrom runner import evaluate, run_suite\n\n\ndef rows(path):\n    with open(path) as handle:\n        return list(csv.DictReader(handle))\n\n\ndef test_four_arms_from_identical_initialization(tmp_path):\n    torch.set_num_threads(1)\n    cfg = Config(n=4,d=64,R=4,steps=7,device="cpu",pair_batch=16)\n    result = Path(run_suite(cfg,tmp_path/"run",seeds=(3,),eval_lags=(1,4,6,7),\n                           prefixes=(0,8),log_every=5,progress=False))\n    runs = rows(result/"runs.csv")\n    assert len(runs)==4\n    assert len({r["initial_hash"] for r in runs})==1\n    assert all(r["status"]=="finite_budget_complete" for r in runs)\n    assert all(r["acquired_positive_gaps"]=="True" for r in runs)\n    hist = rows(result/"history.csv")\n    fixed = [float(r["h"]) for r in hist if r["gate_mode"]=="retrieval_frozen"]\n    assert max(fixed)==min(fixed)\n    assert np.isclose(fixed[0],cfg.h0)\n    assert all(a["passed"] for a in json.loads((result/"native_gradient_audits.json").read_text()))\n    prob = rows(result/"lag_probabilities.csv")\n    assert {int(r["lag"]) for r in prob}=={1,4,6,7}\n    assert all(0<=float(r["min_probability"])<=float(r["mean_probability"])+1e-15<=1+1e-15 for r in prob)\n    for mode in ("learned","retrieval_frozen"):\n        ck = torch.load(result/"checkpoints"/f"seed3_{mode}_adam.pt",weights_only=False)\n        assert ck["optimizer"]["t"]==cfg.steps  # history retained through acquisition\n    with pytest.raises(FileExistsError):\n        run_suite(cfg,result,seeds=(3,),progress=False)\n\n\ndef test_bound_below_finite_prefix_probability():\n    cfg = Config(n=3,d=8,R=2,device="cpu")\n    model = Model(cfg)\n    # This is a mathematical evaluation fixture, never a training initialization.\n    model.Q.zero_(); model.K.zero_()\n    model.Q[:,:3] = torch.eye(3,dtype=torch.float64)*.5\n    model.K.copy_(model.Q)\n    model.theta[0:2] = 4.\n    model.theta[-1] = 3.\n    result = evaluate(model,dict(seed=0,gate_mode="learned",optimizer="sgd"),0,0.,range(1,7),(0,1,16,128))\n    assert all(r["certified_probability_lower_bound"]<=r["min_probability"]+1e-14 for r in result)\n'


In [ ]:
import hashlib
import importlib
import json
import subprocess

for name, source in SOURCES.items():
    (CODE_DIR / (name + ".py")).write_text(source, encoding="utf-8")
    sys.modules.pop(name, None)
sys.path.insert(0, str(CODE_DIR))
importlib.invalidate_caches()
from core import Config
from runner import run_suite
from report import make_report

SOURCE_HASHES = {name + ".py": hashlib.sha256(src.encode()).hexdigest()
                 for name, src in SOURCES.items()}
print("Embedded sources loaded from", CODE_DIR)
show_table(("file", "sha256"), SOURCE_HASHES.items())
if RUN_QA:
    if "test_core" not in SOURCES:
        raise RuntimeError("No embedded test module; regenerate the notebook with test_core.py present.")
    if importlib.util.find_spec("pytest") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "pytest>=7"])
    tests = [str(CODE_DIR / (name + ".py")) for name in SOURCES if name.startswith("test_")]
    test_env = dict(os.environ, PYTEST_DISABLE_PLUGIN_AUTOLOAD="1")
    subprocess.check_call([sys.executable, "-m", "pytest", "-q", *tests],
                          cwd=CODE_DIR, env=test_env)


## Initialization and optimizer settings

Every seed initializes independent Q/K Gaussian tables with standard deviation 10⁻⁶,
q=p=0.5, u=v=1.7 and decoder w=0. The learned retrieval logit is set to
softplus⁻¹(h₀). Three optimizer-specific, answer-supervised acquisition updates acquire
matching; Adam starts with zero moments and retains its full acquisition history.
There is no weight decay, target-position loss, gradient clipping or optimizer-state reset.

Adam uses β₁=0.9, β₂=0.999 (so β₁²<β₂) and epsilon σ³ exp(−c t) outside the square root.
The notebook records the complete configuration. Learning rates differ by optimizer because
the paper prescribes different acquisition and continuation protocols. Equal update counts
are a compute comparison, not equality of optimizer clocks; also inspect the plots versus
cumulative scalar learning rate. No long-lag test result is used to select these rates.

The width is configurable. The default N=8, d=2,048 meets the stated Gaussian dimension
lower bound at failure budget 0.05, but this alone does not verify the theorem's rate,
initialization, cone, settling or basin constants. The smoke width deliberately does not.


In [ ]:
cfg = Config(
    n=N, d=WIDTH, R=R, h0=H0, steps=STEPS, device=DEVICE,
    sigma=1e-6, q0=0.5, u0=1.7,
    beta1=0.9, beta2=0.999,
    lr_sgd=5.0, lr_adam=0.015,
    offset=1000.0, power=0.75,
    table_lr=1e-9, table_power=2.0,
    pair_batch=4096, batch_growth=0.5,
    eps_decay=0.1,
)
GATE_MODES = ("learned", "retrieval_frozen") + (("both_frozen",) if INCLUDE_BOTH_FROZEN else ())
OPTIMIZERS = ("sgd", "adam")
from dataclasses import asdict
show_table(("setting", "value"), asdict(cfg).items())
print("Frozen retention per record:", np.exp(-H0), "| h0 > log(2):", H0 > np.log(2))
print("At R=2: learned-SGD reference boundary is lag 4. At R>2, R+2 is conditional here.")


## Train all requested arms

The runner saves the full weighted population (`dataset/dataset.jsonl`), exactly K base recall
sequences (`dataset/recall_at_R.jsonl`), pair identities (`dataset/pairs.jsonl`), configurations
and diagnostics.
Use a fresh `RUN_TAG` when changing settings; retain the downloaded ZIP or enable Drive for persistence.
Inspect any failed acquisition or numerical status before interpreting a curve.


In [ ]:
if RUN_TRAINING:
    saved_root = run_suite(
        cfg, OUT_DIR, seeds=SEEDS, gate_modes=GATE_MODES,
        optimizers=OPTIMIZERS, eval_lags=EVAL_LAGS,
        prefixes=PREFIXES, log_every=LOG_EVERY,
    )
    OUT_DIR = Path(saved_root)
else:
    print("Training skipped. Set OUT_DIR to a previously generated output folder to plot it.")

# Keep the exact executable code next to the numeric results.
if OUT_DIR.exists():
    audit = OUT_DIR / "embedded_source"
    audit.mkdir(exist_ok=True)
    for name, source in SOURCES.items():
        (audit / (name + ".py")).write_text(source, encoding="utf-8")
    (audit / "sha256.json").write_text(json.dumps(SOURCE_HASHES, indent=2), encoding="utf-8")


## Generalization plots and run diagnostics

The final-lag panels answer the main question: how does correct-answer confidence change as the
target moves farther back? The time/clock panels show whether that range is still expanding.
The solid curves average each seed's minimum over the evaluated key pairs and prefix lengths;
their envelopes span seeds. Dashed curves show mean probabilities over the evaluated examples.
Compare all four arms and inspect the saved prefix-specific measurements.
The theoretical cutoff is a worst-case asymptotic statement; these finite prefix tests are not
a proof over arbitrary streams.


In [ ]:
if not OUT_DIR.exists():
    raise FileNotFoundError(f"No results at {OUT_DIR}; run training or set the correct output path.")
REPORT_PATHS = make_report(OUT_DIR)
print("Report outputs:", REPORT_PATHS)
for path in sorted(OUT_DIR.rglob("*.png")):
    display(Image(filename=str(path), width=1100))
summary_candidates = [p for p in sorted(OUT_DIR.rglob("*.csv"))
                      if "summary" in p.name or p.name == "runs.csv"]
for path in summary_candidates:
    display(Markdown(f"**{path.relative_to(OUT_DIR)}**"))
    import csv
    with path.open() as handle:
        reader = csv.DictReader(handle)
        rows = list(reader)
        columns = list(reader.fieldnames or ())
    if path.name == "runs.csv":
        columns = [name for name in ("seed", "gate_mode", "optimizer", "status", "completed_steps",
                                      "requested_steps", "acquired_positive_gaps", "reason") if name in columns]
    show_table(columns, ([row.get(name, "") for name in columns] for row in rows))
for path in sorted(OUT_DIR.rglob("*.md")):
    if path.name.lower() in {"report.md", "summary.md"}:
        display(Markdown(path.read_text(encoding="utf-8")))


## Export everything

The ZIP contains saved measurements, reports, plots and the exact embedded code. Retain it before
the Colab runtime expires. This notebook itself remains a clean, rerunnable experiment artifact;
save a copy in Colab if you also want its displayed outputs.


In [ ]:
import shutil

zip_path = Path(shutil.make_archive(str(OUT_DIR), "zip", root_dir=OUT_DIR.parent, base_dir=OUT_DIR.name))
print("Saved", zip_path, f"({zip_path.stat().st_size / 2**20:.2f} MiB)")
if IN_COLAB and DOWNLOAD_AT_END:
    from google.colab import files
    files.download(str(zip_path))
else:
    display(FileLink(str(zip_path)))
